Notebook to combine multiple data soruces for Experience Replay on saome arbitrary ratio.

## Libraries and Data Loading

In [2]:
import json
from datasets import load_dataset

from seqeval.metrics.sequence_labeling import get_entities

import re

from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd
from typing import List, Dict, Set
import os

from dataset_processing import *

### Pile-NER

In [2]:
with open("/home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/GLINER_DATA/pilener_train.json", "r") as f:
    pilener = json.load(f)

In [3]:
i = 34
print(" ".join(pilener[i]["tokenized_text"]))
print(pilener[i]["tokenized_text"])
print(pilener[i]["ner"])
print(pilener[i]["negative"])

for entity in  pilener[i]["ner"]:
    temp = pilener[i]["tokenized_text"][entity[0]:entity[1] + 1] # The ranges are inclusive from both sides (thus entity[1] + 1).
    temp_context = pilener[i]["tokenized_text"][max(0, entity[0]-2):min(entity[1]+2, len(pilener[i]["tokenized_text"])-1)]
    print(f"Entry: {entity} ---  pilener[i][\"tokenized_text\"][{entity[0]}:{entity[1]}]: {temp} --- Context: {temp_context}")
    # 
    # 

Cookies help us deliver our Services . By using our Services , you agree to our use of cookies . Learn More Shazam ! character breakdowns confirm film ' s villains Shazam has his work cut out for him . A new batch of character breakdowns for DC ' s impending Shazam ! ( courtesy of That Hashtag Show ) reveal that the titular hero will go up against as many as four villains – including three members of the Sivana Family and a baddie whose whole persona is rooted in mystery . The first antagonist Billy Batson / Shazam , played by Chuck and Thor : Ragnarok actor Zachary Levi , will attempt to take down is his arch-nemesis Doctor Thaddeus Bodog Sivana , an " evil sorcerer who regenerates from death with world domination on his mind . " Within the DC Comics lore , Doctor Sivana has consistently been portrayed as ( unsurprisingly given his title ) a doctor of science , so it ' s interesting to hear that Shazam ! will see him cozying up to the magical side of things as he had done in The New 5

### ClimateIE NER

The data on [GitHub](https://github.com/Jo-Pan/ClimateIE/tree/main/corpus) is incomplete, thus an [issue](https://github.com/Jo-Pan/ClimateIE/issues/1#issue-3406537020) is opened. Until this is solved, experiments will be moved to other datasets: [Climate-Change-NER](https://huggingface.co/datasets/ibm-research/Climate-Change-NER/blob/main/en.test.txt) dataset and [BiodivNER](https://bdj.pensoft.net/article/89481/instance/7788834/) dataset .

In [3]:
# --- Configuration ---
HF_DIRECTORY = "ibm-research/Climate-Change-NER"
LABELS = {
    "climate-assets", "climate-datasets", "climate-greenhouse-gases",     
    "climate-hazards", "climate-impacts", "climate-mitigations",     
    "climate-models", "climate-nature", "climate-observations",     
    "climate-organisms", "climate-organizations", "climate-problem-origins",
    "climate-properties"
}

# --- Initialization ---
print(f"Loading dataset '{HF_DIRECTORY}'...")
# The 'text' feature in this dataset version is what we need.
ds = load_dataset(HF_DIRECTORY) 

Loading dataset 'ibm-research/Climate-Change-NER'...


In [5]:
train_documentwise = []
for i, line in enumerate(ds["train"]["text"]):
    if i == 0:
        temp_list = []
        temp_list.append(line)
        continue
    
    if "-DOCSTART- -X- O O" in line:
        train_documentwise.append(temp_list)
        temp_list = []
        temp_list.append(line)
        continue
    
    temp_list.append(line)



In [6]:
train_documentwise[0]

['-DOCSTART- -X- O O 0b67946e55e9780a2939fcfccbcfb167',
 '',
 'The O',
 'outcomes O',
 'of O',
 'the O',
 'sensitivity O',
 'analysis O',
 'framework O',
 'suggest O',
 'that O',
 'flood B-climate-hazards',
 'risk O',
 'can O',
 'vary O',
 'dramatically O',
 'as O',
 'a O',
 'result O',
 'of O',
 'possible O',
 'change O',
 'scenarios O',
 '. O',
 '',
 'The O',
 'risk O',
 'components O',
 'that O',
 'have O',
 'not O',
 'received O',
 'much O',
 'attention O',
 '( O',
 'e.g. O',
 'changes O',
 'in O',
 'dike B-climate-mitigations',
 'systems O',
 'and O',
 'in O',
 'vulnerability O',
 ') O',
 'may O',
 'mask O',
 'the O',
 'influence O',
 'of O',
 'climate O',
 'change O',
 'that O',
 'is O',
 'often O',
 'investigated O',
 'component O',
 '. O',
 '']

In [7]:
# We will build two master lists for the entire split
all_tokens_in_split = []
all_tags_in_split = []

for document in train_documentwise:
    # 2. Iterate through each line (row) provided by the dataset iterator
    for row in document:
        
        # 3. Parse the 'token tag' string from the 'text' field
        parts = line.rsplit(' ', 1)
        if len(parts) == 2:
            token, tag = parts
            all_tokens_in_split.append(token)
            all_tags_in_split.append(tag)
            
    # 4. After processing all lines, we have the full sequence for the split.
    #    Now, use seqeval to extract entities from this complete sequence.
        print(f"Found {len(all_tokens_in_split)} tokens in the split. Extracting entities...")
        extracted_entities = get_entities(all_tags_in_split)

        # # 5. Reconstruct the entity text and add it to our dictionary
        for entity_label, start_idx, end_idx in extracted_entities:
            
            # Join the tokens to form the full entity string
            entity_text = " ".join(all_tokens_in_split[start_idx : end_idx + 1])
            print(entity_label, " - ", entity_text)
        break
    



Found 1 tokens in the split. Extracting entities...
Found 2 tokens in the split. Extracting entities...
Found 3 tokens in the split. Extracting entities...
Found 4 tokens in the split. Extracting entities...
Found 5 tokens in the split. Extracting entities...
Found 6 tokens in the split. Extracting entities...
Found 7 tokens in the split. Extracting entities...
Found 8 tokens in the split. Extracting entities...
Found 9 tokens in the split. Extracting entities...
Found 10 tokens in the split. Extracting entities...
Found 11 tokens in the split. Extracting entities...
Found 12 tokens in the split. Extracting entities...
Found 13 tokens in the split. Extracting entities...
Found 14 tokens in the split. Extracting entities...
Found 15 tokens in the split. Extracting entities...
Found 16 tokens in the split. Extracting entities...
Found 17 tokens in the split. Extracting entities...
Found 18 tokens in the split. Extracting entities...
Found 19 tokens in the split. Extracting entities...
Fo

### Climate-Change-NER

In [8]:
# --- Configuration ---
HF_DIRECTORY = "ibm-research/Climate-Change-NER"
LABELS = {
    "climate-assets", "climate-datasets", "climate-greenhouse-gases",     
    "climate-hazards", "climate-impacts", "climate-mitigations",     
    "climate-models", "climate-nature", "climate-observations",     
    "climate-organisms", "climate-organizations", "climate-problem-origins",
    "climate-properties"
}

# 1. Load and pre-process the data into a document-wise list of lines
print(f"Loading and splitting dataset '{HF_DIRECTORY}' by document...")
ds = load_dataset(HF_DIRECTORY) 
train_documentwise = []
temp_list = []
for line in ds["train"]["text"]:
    if line.strip().startswith('-DOCSTART-'):
        if temp_list:
            train_documentwise.append(temp_list)
        temp_list = []
    temp_list.append(line)
if temp_list:
    train_documentwise.append(temp_list)

# 2. Run the first function to get structured documents with CHARACTER spans
structured_documents_char_spans = ibmccner_process_bio_documents(
    document_list=train_documentwise,
    labels_to_keep=LABELS
)

# 3. Run the second function to convert the result into TOKEN spans
ibmccner = convert_to_token_spans(structured_documents_char_spans)


Loading and splitting dataset 'ibm-research/Climate-Change-NER' by document...


In [9]:
ibmccner

[{'tokenized_text': ['The',
   'outcomes',
   'of',
   'the',
   'sensitivity',
   'analysis',
   'framework',
   'suggest',
   'that',
   'flood',
   'risk',
   'can',
   'vary',
   'dramatically',
   'as',
   'a',
   'result',
   'of',
   'possible',
   'change',
   'scenarios',
   '.',
   'The',
   'risk',
   'components',
   'that',
   'have',
   'not',
   'received',
   'much',
   'attention',
   '(',
   'e',
   '.',
   'g',
   '.',
   'changes',
   'in',
   'dike',
   'systems',
   'and',
   'in',
   'vulnerability',
   ')',
   'may',
   'mask',
   'the',
   'influence',
   'of',
   'climate',
   'change',
   'that',
   'is',
   'often',
   'investigated',
   'component',
   '.'],
  'ner': [[9, 9, 'climate-hazards'], [38, 38, 'climate-mitigations']]},
 {'tokenized_text': ['A',
   'parameterization',
   'of',
   'vertical',
   'diffusivity',
   'in',
   'ocean',
   'general',
   'circulation',
   'models',
   'has',
   'been',
   'implemented',
   'in',
   'the',
   'ocean',
   'm

### BiodivNER

In [11]:

DATA_DIRECTORY = "/home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER"

EXPECTED_LABELS = {
    'Quality', 'Phenomena', 'Organism',
    'Matter', 'Location', 'Environment'
}

# --- Execute the function and inspect the results ---
biodivner_structured_data = biodivner_process_bio_documents(
    data_dir=DATA_DIRECTORY,
    labels_to_keep=EXPECTED_LABELS
)

# 3. Run the second function to convert the result into TOKEN spans
biodivner = convert_to_token_spans(biodivner_structured_data)


Processing file: /home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER/dev.csv...
Processing file: /home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER/train.csv...


In [12]:
len(biodivner)

2158

### CWED4ETA

In [ ]:
DIR = "/home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/CWED4ETA/CWED4ETA.conll"

CWED4ETA_LABELS = {
    "Ecosystem", "Energy Source", "Natural Disaster", 
    "Meteorological Phenomenon", "Quantity", "Astronomical Object", 
    "Body of Water", "Disease", "Location", 
    "Physical Phenomenon", "Chemical", "Time Period", 
    "Organization", "Natural Phenomenon", "Field of Study", 
    "Mathematical Expression", "Measuring Device", "Geographical Feature", 
    "System", "Satellite", "Organism", 
    "Method", "Other", "Person", 
    "Artefact", "Body Part", "Symptom"
}

## Data Stats

In [13]:
# len(pilener)
# len(ibmccner)
len(biodivner)

print("Total number of tagged documents/sentences: ", len(biodivner))
print("Values associated with each document/sentence: ", biodivner[0].keys())
print("Example of \'tokenized_text\': ", biodivner[0]["tokenized_text"])
print("Example of \'ner\': ",biodivner[0]["ner"])

Total number of tagged documents/sentences:  2158
Values associated with each document/sentence:  dict_keys(['tokenized_text', 'ner'])
Example of 'tokenized_text':  ['For', 'live', 'snags', 'the', 'measurement', 'height', 'is', 'indicated', 'by', 'a', 'white', 'mark', 'made', 'by', 'Martin', 'Barrufol', '-', 'see', 'dbh', 'for', 'this', ')', '.']
Example of 'ner':  [[1, 2, 'Organism'], [5, 5, 'Quality']]


In [ ]:
def calculate_avg_token_length(dataset: List[Dict]) -> float:
    """
    Calculates the average length of the 'tokenized_text' in a dataset.
    """
    if not dataset:
        return 0.0
    
    total_tokens = sum(len(doc['tokenized_text']) for doc in dataset)
    return total_tokens / len(dataset)

def calculate_avg_entity_count(dataset: List[Dict]) -> float:
    """
    Calculates the average number of entities per document/sentence in a dataset.
    """
    if not dataset:
        return 0.0
        
    total_entities = sum(len(doc['ner']) for doc in dataset)
    return total_entities / len(dataset)

def calculate_entity_distribution(dataset: List[Dict]) -> Counter:
    """
    Calculates the frequency of each entity type across the entire dataset.
    
    Returns:
        A collections.Counter object mapping entity labels to their frequencies.
    """
    all_labels = [entity[2] for doc in dataset for entity in doc['ner']]
    return Counter(all_labels)

def plot_entity_distribution(distribution: Counter, title: str = "Entity Type Distribution"):
    """
    Creates and displays a bar chart of entity type frequencies.
    """
    # Convert the Counter to a pandas Series for easy sorting and plotting
    dist_series = pd.Series(distribution).sort_values(ascending=False)
    
    plt.figure(figsize=(12, 8))
    dist_series.plot(kind='bar')
    
    plt.title(title, fontsize=16)
    plt.ylabel('Frequency', fontsize=12)
    plt.xlabel('Entity Type', fontsize=12)
    plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
    plt.tight_layout() # Adjust layout to prevent labels from overlapping
    plt.show()

def plot_length_distribution(dataset: List[Dict], title: str = "Distribution of Document Length"):
    """
    Calculates and plots a histogram of the document lengths based on 'tokenized_text'.

    Args:
        dataset: A list of dictionaries, where each dict has a 'tokenized_text' key.
        title: The title for the plot.
    """
    if not dataset:
        print("Dataset is empty. Cannot generate plot.")
        return

    # 1. Extract the length of each document's tokenized_text
    lengths = [len(doc['tokenized_text']) for doc in dataset]

    # 2. Create the histogram plot
    plt.figure(figsize=(10, 6))
    
    # Use a reasonable number of bins, e.g., 50.
    # The 'edgecolor' makes the bars more distinct.
    plt.hist(lengths, bins=50, edgecolor='black')
    
    # 3. Add informative labels and a title
    plt.title(title, fontsize=16)
    plt.xlabel("Number of Tokens per Document", fontsize=12)
    plt.ylabel("Frequency (Number of Documents)", fontsize=12)
    plt.grid(axis='y', alpha=0.75) # Add a grid for easier reading of frequencies
    
    # 4. Display the plot
    plt.tight_layout()
    plt.show()


DATA = pilener

# 0. Total number of documents
print("Total number of documents: ", len(DATA))

# 1. Calculate the average length of tokenized text
avg_len = calculate_avg_token_length(DATA)
print(f"Average tokenized text length: {avg_len:.2f} tokens per document")

# 2. Calculate the average number of entities
avg_entities = calculate_avg_entity_count(DATA)
print(f"Average number of entities: {avg_entities:.2f} entities per document")

# 2.5 Print 


# 3. Calculate and plot the entity type distribution
entity_dist = calculate_entity_distribution(DATA)
print("\nEntity Type Distribution (Frequencies):")
print(entity_dist)

# # This will open a window with the bar chart
plot_entity_distribution(entity_dist, title="IBM-CC-NER Entity Distribution")

plot_length_distribution(DATA, title="IBM-CC-NER Distribution of Document Length")

Total number of documents:  45889
Average tokenized text length: 202.95 tokens per document
Average number of entities: 20.48 entities per document

Entity Type Distribution (Frequencies):
Counter({'concept': 43743, 'Person': 40263, 'person': 38936, 'Organization': 38259, 'Location': 32023, 'organization': 31526, 'product': 28823, 'location': 28170, 'variable': 21974, 'Concept': 15370, 'object': 15115, 'Product': 12304, 'technology': 11391, 'Date': 10597, 'chemical': 9881, 'Medical Condition': 9812, 'software': 9655, 'number': 9606, 'medical condition': 9477, 'disease': 7869, 'attribute': 7560, 'date': 7347, 'Other': 6599, 'Technology': 6482, 'entity type': 6394, 'group': 6346, 'function': 6225, 'measurement': 6128, 'event': 5905, 'class': 5880, 'material': 5741, 'protein': 5343, 'Event': 4987, 'process': 4687, 'Nationality': 4040, 'condition': 3923, 'substance': 3866, 'Country': 3865, 'type': 3753, 'animal': 3530, 'component': 3420, 'activity': 3324, 'method': 3293, 'company': 3282, '

KeyboardInterrupt: 